<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex12.2_39_edge-realtime-screening/Ex12.2_39_edge_realtime_screening.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_12.2 (39 buses, edge) — Real-Time Stability Screening on the Edge

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L12.2 · Prediction of Power Grid Stability**

Every 20 ms the phasor measurement units (PMUs) of a grid send a new frame:
the angle and frequency at every bus. A control room wants to know, **for
every fault that could happen next, whether the grid would stay in step** -
and to know it before the next frame arrives.

You build that screen for the IEEE 39-bus grid of Ex_12.1 and 35 credible
faults. The reference is the simulation: RK4 on the third-order model. The
fast part of the model, the load buses following their neighbours within
milliseconds, holds an explicit integrator to 10 ms steps. A **graph-network
surrogate of the dynamics** steps 50 ms at a time, trained on simulated
trajectories with the machines' equations as a physics term. Then the screen
runs in real time: a PMU player and a screener talking over the network,
the same two scripts that run on a Jetson AGX Orin and a Jetson AGX Thor in
the mini projects - here on a **virtual board**, in Colab.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the problem, its data and its physics | — |
| **4** | the reference, and why its step cannot grow | — |
| **5** | a graph-network surrogate of the dynamics | the physics residual |
| **6** | the screen, compared with the reference | the batched screen |
| **7** | the real-time loop on a virtual board | — |
| **8** | what the notebook says | — |
| **9 – 10** | your report, and the mini project proposal | your answers |

Use a GPU runtime (Runtime → Change runtime type → T4 GPU): the timings of sections 6 and 7 are meant for it. On a
CPU it takes about four minutes, and the timings come out differently (section 6 says how). The trained surrogate is fetched with the
libraries; set `FULL_TRAINING = True` in section 5 to train it again
(about two minutes on a CPU). Numbers are printed with at most two decimals.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py', 'grid39.py', 'Ex12.2_39_edge_surrogate.pt', 'board/frames.py', 'board/surrogate.py', 'board/pmu_player.py', 'board/screener.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex12.2_39_edge-realtime-screening/"
ON_COLAB = "google.colab" in sys.modules
os.makedirs("board", exist_ok=True)
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "problem", "grid39", "frames", "surrogate", "pmu_player", "screener"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse as sp, scipy.sparse.linalg as spla

cc.OUTPUT_DIR = "Ex12.2_39_edge_outputs"                              # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))
import sys; sys.path.insert(0, "board")
import frames, surrogate, pmu_player, screener                # the board scripts: packets, the surrogate, the two ends of the loop
WEIGHTS = "Ex12.2_39_edge_surrogate.pt"


---

## 1 · The problem

**N-1 security** means the grid survives the loss of any one element. The
dangerous moment is a fault: a short circuit on a branch, cleared by the
protection tripping that branch. During the fault the machines near it
cannot deliver their power and speed up; after it, on a network with one
branch fewer, they must slow down again. If one has gone too far it slips a
pole and loses step.

**The screen** (L12.2's *Real-Time Screening*): for the grid as the latest
PMU frame shows it, simulate every credible fault and flag the ones the grid
would not survive. Here there are 35 - a solid fault at the first bus of
each branch whose loss leaves the grid whole, cleared after 150 ms - and the
answer is wanted within one frame, 20 ms.

## 2 · The data

| | |
|---|---|
| grid | IEEE 39-bus, the model of Ex_12.1: machines third order, converters second order, load buses first order |
| contingencies | 35: a solid fault at the first bus of a branch, cleared after 150 ms by tripping the branch |
| verdict | **insecure** if within 3 s some machine moves more than 180° from the grid's centre of inertia |
| operating states | load 0.9 to 1.15 of the case; up to four of the nine local machines replaced by converters (share up to 45 %) |
| training | 60 states × 35 contingencies = 2 100 simulated trajectories; testing on 15 other states |
| PMU frames | 50 per second; angle noise 0.01 rad |
| assumed, not data | as in Ex_12.1: a load bus follows its neighbours within 10 ms; machines' and converters' damping; converters' inertia 2 s |

## 3 · The physics

The model of Ex_12.1, for every bus $i$ (L12.2's *Third-Order Model*):

$$\frac{d\delta_i}{dt} = \omega_i, \qquad
  M_i\frac{d\omega_i}{dt} = P_i - D_i\,\omega_i - P_{e,i}, \qquad
  T_i\frac{dE_i}{dt} = E_{f,i} - E_i + X_i\sum_j B_{ij}E_j\cos(\delta_i-\delta_j)$$

for the machines (converters keep $E_i$), and $D_i\,d\delta_i/dt = P_i -
P_{e,i}$ for a load bus, with $P_{e,i} = \sum_j E_iE_jB_{ij}\sin(\delta_i -
\delta_j)$. **During the fault** the faulted bus is held at zero voltage: it
exchanges no power. **After it**, $B$ is the network without the tripped
branch.

**The verdict** uses the centre of inertia, $\delta_{\mathrm{COI}} = \sum_i
M_i\delta_i / \sum_i M_i$ over the machines and converters: a machine more
than $\pi$ from it has lost step.

The cell finds a fault that the base case survives and another operating
state does not, draws it on the grid, and simulates it in both.

In [ ]:
pb.describe_problem()
states = pb.operating_states(75, seed=12)                     # 75 operating states: 60 to train on, 15 to test
for g in states: g.to(DEVICE, torch.float32)                  # each state's arrays as float32 tensors on the device
train_states, test_states = states[:60], states[60:]

g0 = pb.Grid().to(DEVICE, torch.float32)                      # the base case: no converters, the load of the case
traj0, worst0 = pb.reference(g0)
for j, g in enumerate(states):                                # a state in which some fault, harmless in the base case, is not
    traj1, worst1 = pb.reference(g)
    flip = np.flatnonzero((worst1.cpu().numpy() > np.pi) & (worst0.cpu().numpy() < np.pi))
    if len(flip):
        case = int(flip[0]); break
fb, br = pb.CASES[case]
print(f"  fault at bus {fb + 1}, cleared by tripping branch {pb.LINES[br][0] + 1}-{pb.LINES[br][1] + 1}:")
print(f"    base case: the machines swing to {np.degrees(float(worst0[case])):.2f} deg and come back")
sep1 = pb.separation(traj1[case, :, :, 0], pb.inertia_weights(g)).cpu().numpy()
t_cross = pb.T_CLEAR + pb.DT_NET * int(np.argmax(sep1 > np.pi))
print(f"    state {j} (load {g.load:.2f}, {100 * g.renewable_share:.0f} % converters): a machine passes 180 deg "
      f"{t_cross:.2f} s after the fault and keeps slipping")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
pb.draw_grid(grid=pb.Grid(out=(br,)), ax=ax[0], title=f"fault at bus {fb + 1}, cleared by tripping the dotted branch")
ax[0].scatter(*np.array(pb.XY)[fb], s=500, facecolors="none", edgecolors=pb.ORANGE, linewidths=2.2)
t = pb.T_CLEAR + pb.DT_NET * np.arange(traj0.shape[1])
for gg, traj, name, style in ((g0, traj0, "base case: secure", "-"), (g, traj1, f"state {j}: insecure", "--")):
    ax[1].plot(t, np.degrees(pb.separation(traj[case, :, :, 0], pb.inertia_weights(gg)).cpu().numpy()), style, lw=2, label=name)
ax[1].axhline(180, color="0.5", lw=1); ax[1].set_ylim(0, 400)
ax[1].set_xlabel("time after the fault  [s]"); ax[1].set_ylabel("largest machine angle from the COI  [deg]"); ax[1].legend()
plt.tight_layout(); plt.show()

**What you should see.** The same fault in two operating states: in the base
case the machines swing to 72.13° and come back; in a state with
40 % of the local power from converters a machine passes 180° and
keeps slipping. **The
verdict depends on the state the grid is in**, which is why the screen must
be run again for every frame.

---

## 4 · The reference, and why its step cannot grow

**The reference** is RK4 on a 10 ms step, for 3 s: 15 steps of fault, then
285 on each case's own network. All 35 cases run as one batch.

**The classical saving would be a longer step.** It is not available here.
A load bus follows its neighbours within about 10 ms; that makes the model
*stiff*: its fastest eigenvalue is about −196 1/s, and an explicit method like
RK4 is stable only while the step times that eigenvalue stays above −2.8. RK4
on 20 ms does not lose accuracy - it blows up. (An implicit method could take
a longer step, at the price of solving equations at every one.)

The cell computes the reference for all 75 states, checks it against a 5 ms
step, tries a 20 ms step, and times one state's screen.

In [ ]:
sync = (lambda: torch.cuda.synchronize()) if DEVICE.type == "cuda" else (lambda: None)
ref = [pb.reference(g) for g in states]                       # every state, every contingency, RK4 on 10 ms
traj_all = [r[0] for r in ref]; worst_ref = torch.stack([r[1] for r in ref]).cpu().numpy()
insecure = worst_ref > np.pi
print(f"  {insecure.size} cases: {insecure.sum()} insecure ({100 * insecure.mean():.1f} %); "
      f"training {insecure[:60].sum()}, testing {insecure[60:].sum()}")

fine = np.stack([pb.reference(g, dt=0.005)[1].cpu().numpy() for g in test_states])
print(f"  5 ms against 10 ms on the 15 test states: {int(((fine > np.pi) != insecure[60:]).sum())} verdicts differ, "
      f"largest separation differs by {np.degrees(np.abs(fine - worst_ref[60:])[fine < np.pi].max()):.2f} deg at most (secure cases)")
for dt in (0.02, 0.025):                                      # longer steps: the classical saving
    w = np.stack([pb.reference(g, dt=dt)[1].cpu().numpy() for g in test_states])
    flips = int(((w > np.pi) != insecure[60:]).sum())
    sec = (worst_ref[60:] < np.pi) & np.isfinite(w)
    print(f"  {1000 * dt:.0f} ms: {flips} of 525 verdicts differ; on secure cases the separation is off by up to "
          f"{np.degrees(np.abs(w - worst_ref[60:])[sec]).max():,.0f} deg")

def rk4_screen(g, z0=None):                                   # one state's screen by the reference
    z = pb.fault_phase(g, z0)
    z = pb.after_clearing(g, z, pb.post_networks(g))
    return z
t_rk4 = middle(lambda: (rk4_screen(test_states[0]), sync()))
print(f"  one state's screen by RK4 on 10 ms: {1000 * t_rk4:.2f} ms on this runtime (35 cases, 300 steps)")

**What you should see.** 108 of the 2 625 cases are insecure
(4.1 %). The 5 ms step changes no verdict. **At 20 ms 2
verdicts change and some secure cases come out 15 654° off; at 25 ms
432 of 525 change**: past its stability limit (about 14 ms here) RK4
does not lose accuracy gradually, it goes unstable. One state's screen takes about 120 ms on a CPU - six times the 20 ms of a
frame.

---

## 5 · A graph-network surrogate of the dynamics

**The surrogate** (L12.2's *Learning the Dynamics*) replaces RK4 after the
fault is cleared. It is a graph network that takes the state of every bus and
returns its change over 50 ms:

$$x_i(t+\Delta t) = x_i(t) + s \odot \mathcal{N}_\theta\big(\hat{A}, H(t)\big)_i, \qquad \Delta t = 50\ \mathrm{ms}$$

with $x_i = (\delta_i, \omega_i, E_i)$, $s$ the typical change of each in
50 ms, $\hat{A}$ the network after the branch is tripped (the graph layer of
Ex_12.1), and $H$ twelve numbers per bus: the angle from the centre of
inertia (as its sine and cosine), the frequency, $E_i$, and eight that do not
change - power, inertia, damping, whether it is a machine, $x_d - x'_d$,
$T'_{d0}$, $E_f$, $E_0$. **The fault itself stays with RK4**: 150 ms, 15
steps, then the surrogate takes over from the state at clearing - 57 steps
for the 2.85 s that remain, against 285 for RK4.

**The loss** has two parts. The **data** part is the distance from the
reference trajectories over six steps of the surrogate run in a row (300 ms),
with the insecure cases weighted five times, since they are few and they
matter most. The **physics** part asks that every step satisfy the
machines' equations, integrated by the trapezoidal rule:

$$r_\delta = \delta^{k+1} - \delta^k - \tfrac{\Delta t}{2}\big(\omega^k + \omega^{k+1}\big), \qquad
  r_\omega = \omega^{k+1} - \omega^k - \tfrac{\Delta t}{2}\big(f_\omega^k + f_\omega^{k+1}\big), \qquad
  r_E = E^{k+1} - E^k - \tfrac{\Delta t}{2}\big(f_E^k + f_E^{k+1}\big)$$

with $f_\omega$ and $f_E$ the right-hand sides of section 3 evaluated on the
surrogate's own states. The trapezoidal rule is accurate for the machines,
whose swings (0.5 to 1.4 Hz) last 15 or more steps; it is applied at the
machines only, because the load buses move faster than a 50 ms step can
follow.

**TODO 1** is the physics residual.

In [ ]:
def physics_residual(g, x0, x1, B_post):
    # the machines' equations, by the trapezoidal rule, over one surrogate step from x0 to x1 (cases, 39, 3)
    f0 = pb.rhs_batched(g.t, (x0[..., 0], x0[..., 1], x0[..., 2]), B=B_post)   # the right-hand sides at the start of the step
    f1 = pb.rhs_batched(g.t, (x1[..., 0], x1[..., 1], x1[..., 2]), B=B_post)   # ... and at the end, on the surrogate's state
    machine = g.t["field"] > 0                                # the ten machines
    # TODO 1: the three trapezoidal residuals of section 5 for delta, omega and E (i = 0, 1, 2),
    # then their mean square at the machines, each divided by its typical change SCALE[i].
    #
    #   r = [x1[..., i] - x0[..., i] - pb.DT_NET / 2 * (f0[i] + f1[i]) for i in range(3)]
    #   return sum(((r[i] / SCALE[i])[..., machine] ** 2).mean() for i in range(3))
    raise NotImplementedError("Write the physics residual")

In [ ]:
FULL_TRAINING = False                                         # True: train from scratch (about two minutes on a CPU); False: load the course's weights, train a little
SCALE = pb.SCALE.to(DEVICE)                                   # the typical change in 50 ms, as a tensor on the device
def prepared(g):                                              # what one state's training needs, built once
    return dict(g=g, A=pb.graphs(g), stat=pb.statics(g), Mw=pb.inertia_weights(g), B=pb.post_networks(g))
TRAIN_SET = [dict(prepared(g), traj=traj_all[i], bad=torch.as_tensor(insecure[i], device=DEVICE)) for i, g in enumerate(train_states)]

def unroll(net, s, x, steps):                                 # the surrogate run `steps` times from x
    out = []
    for _ in range(steps):
        x = surrogate.step(net, x, s["A"], s["stat"], s["Mw"], SCALE)
        out.append(x)
    return torch.stack(out, 1)

U, W_PHYSICS = 6, 0.1
def loss_fn(net, s, k0):
    traj = s["traj"]
    xs = unroll(net, s, traj[:, k0], U)                       # six steps from the reference state at step k0
    weight = (1 + 4 * s["bad"].float())[:, None, None, None]  # an insecure case counts five times
    data = (weight * ((xs - traj[:, k0 + 1:k0 + 1 + U]) / SCALE) ** 2).mean()
    prev = torch.cat([traj[:, k0:k0 + 1], xs[:, :-1]], 1)
    phys = sum(physics_residual(s["g"], prev[:, j], xs[:, j], s["B"]) for j in range(U)) / U
    return data + W_PHYSICS * phys, data, phys

net = surrogate.Surrogate().to(DEVICE, torch.float32)         # the surrogate network, on the device, in float32
set_seed(88)                                                  # fix the random start so a run repeats
if not FULL_TRAINING:
    net.load_state_dict(torch.load(WEIGHTS, map_location=DEVICE))   # the course's trained weights
steps, lr = (4000, 2e-3) if FULL_TRAINING else (300, 2e-4)
opt = torch.optim.Adam(net.parameters(), lr=lr)               # the Adam optimiser
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)   # the step size falls to zero along a cosine
n_frames = traj_all[0].shape[1]
t0 = time.perf_counter()
for it in range(steps):
    s = TRAIN_SET[it % len(TRAIN_SET)]
    k0 = int(np.random.randint(0, n_frames - U))
    loss, data, phys = loss_fn(net, s, k0)
    opt.zero_grad(); loss.backward()                          # clear old gradients; backpropagate
    torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)     # cap the gradient's length
    opt.step(); sched.step()                                  # one Adam step; the next step size
    if it % (steps // 4) == 0 or it == steps - 1:
        print(f"    step {it:5d}   loss {loss.item():.3e}   data {data.item():.3e}   physics {phys.item():.3e}")
train_time = time.perf_counter() - t0
if FULL_TRAINING:
    torch.save(net.state_dict(), WEIGHTS)                     # the trained weights, for section 7 and the boards
print(f"{parameter_count(net)} weights; {steps} steps in {train_time:.0f} s (offline, not counted below)")

**What you should see.** The loss falling, its data part well above its
physics part. With the course's weights the surrogate is already trained,
and the 300 steps change it little.

---

## 6 · The screen, compared with the reference

**The screen for one PMU frame** is RK4 through the 150 ms of each fault,
then the surrogate for 57 steps on each case's network, keeping each case's
largest separation from the centre of inertia. All 35 cases are one batch:
one graph per case, the 39 buses of each a row.

**TODO 2** is that loop.

In [ ]:
@torch.no_grad()                                              # no gradient tracking: evaluation only
def screen(net, s, z0=None):
    # every contingency from the state z0 (the equilibrium by default): the largest separation of each [rad]
    z = pb.fault_phase(s["g"], z0)                            # the fault, by RK4: 150 ms
    x = torch.stack(z, -1)                                    # (cases, 39, 3)
    worst = pb.separation(x[..., 0], s["Mw"])
    # TODO 2: 57 steps of the surrogate, each followed by the update of every case's largest separation.
    #
    #   for _ in range(int(round((pb.T_HORIZON - pb.T_CLEAR) / pb.DT_NET))):
    #       x = surrogate.step(net, x, s["A"], s["stat"], s["Mw"], SCALE)   # one 50 ms step of the surrogate
    #       worst = torch.maximum(worst, pb.separation(x[..., 0], s["Mw"]))
    raise NotImplementedError("Write the screening loop")
    return worst

In [ ]:
TEST_SET = [prepared(g) for g in test_states]
net.eval()                                                    # evaluation mode
pred = np.stack([screen(net, s).cpu().numpy() for s in TEST_SET])
truth = worst_ref[60:]
bad, flag = truth > np.pi, pred > np.pi
caught, missed, alarms = int((bad & flag).sum()), int((bad & ~flag).sum()), int((~bad & flag).sum())
ok_sec = ~bad & ~flag
err = np.degrees(np.abs(pred - truth)[ok_sec])
t_net = middle(lambda: (screen(net, TEST_SET[0]), sync()))

print(f"{'15 test states, 525 cases':34s}{'insecure caught':>17s}{'missed':>8s}{'false alarms':>14s}{'one state':>12s}")
print(f"  {'RK4, 10 ms (the reference)':32s}{bad.sum():>11d} of {bad.sum():<3d}{0:>8d}{0:>14d}{1000 * t_rk4:9.2f} ms")
print(f"  {'graph-network surrogate':32s}{caught:>11d} of {bad.sum():<3d}{missed:>8d}{alarms:>14d}{1000 * t_net:9.2f} ms")
print(f"  largest separation, secure cases: mean error {err.mean():.2f} deg, worst {err.max():.2f} deg")
print(f"  the surrogate is {t_rk4 / t_net:.1f} times faster for one state, on the same runtime")

fig, ax = plt.subplots(figsize=(5.4, 4.4))
ax.plot([0, 360], [0, 360], "k--", lw=1)
ax.plot(np.degrees(np.clip(truth, 0, 2 * np.pi)).ravel(), np.degrees(np.clip(pred, 0, 2 * np.pi)).ravel(), ".", ms=4)
ax.axvline(180, color="0.5", lw=1); ax.axhline(180, color="0.5", lw=1)
ax.set_xlabel("largest separation, reference  [deg]"); ax.set_ylabel("the surrogate's  [deg]")
ax.set_title("525 held-out cases (clipped at 360 deg)", fontsize=9); plt.tight_layout()
plt.savefig(cc.output_path("Ex12.2_39_edge_report.png"), dpi=150); plt.show()

**What you should see.** The surrogate catches 21 of the 21
insecure cases with 6 false alarms, and on the secure ones it is
3.82° off on average. On a CPU one state's screen takes about 70 ms against
about 130 ms for RK4: **1.8 times faster on the same runtime**, the training
done offline. The gain is small on a CPU: the surrogate takes five times
fewer steps than RK4 but does more arithmetic in each. On a GPU the number
of steps is what costs, and the gain should be larger - run it and see.

**What the physics term did**: trained from scratch without it (`W_PHYSICS = 0`), the surrogate caught the
same 21 cases with the same 6 false alarms, but on the secure cases it was
off by 8.34° on average and 64.67° at worst, against 4.47° and 25.05° when
trained from scratch with it - about twice the error. The physics
term does not change the verdicts here; it makes the trajectories behind
them closer to the model's. (One training run each, on a CPU.)

---

## 7 · The real-time loop on a virtual board

**On the boards** (the mini projects) a Jetson AGX Orin plays the PMUs: it
sends a frame every 20 ms over the network. A Jetson AGX Thor screens: it
takes the newest frame, runs the 35 cases, and answers with the verdicts. The
two scripts are `board/pmu_player.py` and `board/screener.py`.

**Here** both run on the Colab runtime, in two threads, talking over UDP to
the machine itself: the same code, the same packets, the same 20 ms clock.
The player replays a 30 s recording of the grid - the load drifting by a few
per cent, two converters' output swinging with the wind, noise on every
reading. It measures the round trip of every frame; the screener measures its
own processing time and counts the frames it had to drop because a newer one
had arrived.

**What carries over to the boards** is the loop, the code and the ratios. The
milliseconds do not: a Colab GPU is not a Thor, and both threads share one
machine. The mini projects measure them on the boards.

In [ ]:
import threading
torch.save(net.state_dict(), "Ex12.2_39_edge_rt.pt")                  # the weights the screener loads
rec, t_rec = pb.recording(seconds=30.0, device=DEVICE)        # 30 s of PMU frames, 50 per second
np.savez("Ex12.2_39_edge_recording.npz", frames=rec, t=t_rec)         # the file the Orin replays in the mini project
g_rt = pb.Grid(1.0, (), (31, 33)).to(DEVICE, torch.float32)   # the operating state of the recording
rt_net = surrogate.load("Ex12.2_39_edge_rt.pt", DEVICE)               # the screener's own copy of the surrogate

result = {}
srv = threading.Thread(target=lambda: result.update(screen=screener.serve(rt_net, g_rt, 4712, ("127.0.0.1", 4713))))
srv.start(); time.sleep(3.0)                                  # the screener warms up first
result["play"] = pmu_player.play(rec, ("127.0.0.1", 4712), listen_port=4713)
srv.join()

rt, proc = result["play"]["round_trip_ms"], result["screen"]["processing_ms"]
print(f"  frames sent {result['play']['sent']}, answered {len(rt)}, dropped by the screener {result['screen']['dropped']}")
print(f"  screener's processing: median {np.median(proc):.2f} ms, 99th percentile {np.percentile(proc, 99):.2f} ms")
print(f"  round trip:            median {np.median(rt):.2f} ms, 99th percentile {np.percentile(rt, 99):.2f} ms; "
      f"over the 20 ms frame: {(rt > 20).sum()} of {len(rt)}")
print(f"  insecure contingencies flagged per frame: {np.bincount(result['play']['insecure']).tolist()} (frames with 0, 1, 2, ...)")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].plot(result["play"]["seq"] * 0.02, rt, ".", ms=2, label="round trip")
ax[0].axhline(20, color=pb.ORANGE, lw=1.5, label="one frame, 20 ms")
ax[0].set_xlabel("time  [s]"); ax[0].set_ylabel("[ms]"); ax[0].legend()
ax[1].hist(proc, bins=40); ax[1].axvline(20, color=pb.ORANGE, lw=1.5)
ax[1].set_xlabel("screener's processing per frame  [ms]"); ax[1].set_ylabel("frames")
plt.tight_layout(); plt.show()

**What you should see.** The screener answers 376 of the
1 500 frames and drops 1 124; its processing takes 75 ms
(median) on this runtime, and the round trip 85 ms. These are a CPU runtime's numbers: the screen takes longer than a
frame, so three frames in four arrive while the last is still being
screened and are dropped, and every answer is late. On a GPU runtime the
surrogate's 57 steps cost far less than RK4's 300, because a GPU's time goes
into the number of steps rather than the arithmetic in each; whether one
frame's screen fits in 20 ms is the question MP12.2A-39 answers on the Thor. In the
recording the grid is secure against most contingencies, and a few are
flagged in every frame - the same faults as in section 6 at that load.

---

## 8 · What the notebook says

* **The screen is a simulation per contingency, and it must be repeated for
  every state of the grid.** The same fault was harmless in one state and
  fatal in another.
* **The reference cannot take longer steps.** The load buses make the model
  stiff, and explicit RK4 blows up beyond about 14 ms.
* **The surrogate steps 50 ms** because it was trained on the slow motion of
  the machines, which is what decides the verdict. It is not a solver: it is
  only as good as the trajectories it learned from, in the states it saw.
* **The physics term ties every step to the machines' equations.**
  Without it the verdicts were the same and the error on the secure cases twice as large.
* **On the same runtime the surrogate was 1.8 times faster** for one
  frame's screen, and caught 21 of 21 insecure cases with
  6 false alarms.
* **Real time is a property of the whole loop**: the screen, the network
  between the boards, and what happens to a frame that arrives while the last
  is still being screened. The virtual board shows the loop; the boards give
  the milliseconds.
* **The model is the classical one**, with assumed load and converter
  parameters: a converter that limits its current in a fault, or a
  phase-locked loop that loses track, is not in it. Mini project B adds them.

---

## 9 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex12.2_39_edge_report.md` with the results of section 6 and 7 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L12.2's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "The same fault was secure in one operating state and insecure in another. What made the difference, and why must a real-time screen be run again for every PMU frame? (-> L12.2 Q1)": """
""",
    "Why can RK4 not take a 20 ms step on this model, and why can the surrogate take 50 ms? Say what the surrogate does not learn, and what the physics term in your TODO 1 adds to the data. (-> L12.2 Q2)": """
""",
    "Report the surrogate's caught, missed and false alarms, and its time against RK4 on the same runtime. Which of the two numbers would you trust on the Thor, and which would you have to measure again? (-> L12.2 Q3)": """
""",
    "To conclude: would you let this screener raise an alarm in a control room, and would you let it clear one? Say what you would test on the Orin and the Thor before either. (-> L12.2, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| 15 test states, 525 cases | insecure caught | missed | false alarms | one state |", "|---|---|---|---|---|",
            f"| RK4, 10 ms (reference) | {int(bad.sum())} of {int(bad.sum())} | 0 | 0 | {1000 * t_rk4:.2f} ms |",
            f"| graph-network surrogate | {caught} of {int(bad.sum())} | {missed} | {alarms} | {1000 * t_net:.2f} ms |",
            "", f"Largest separation, secure cases: mean error {err.mean():.2f} deg, worst {err.max():.2f} deg.",
            "", f"Virtual board: {len(rt)} of {result['play']['sent']} frames answered, {result['screen']['dropped']} dropped; "
            f"processing median {np.median(proc):.2f} ms, round trip median {np.median(rt):.2f} ms.",
            "", "Load and converter parameters are assumed; the milliseconds are this runtime's, not the boards'."]
    out = ["# Ex_12.2_39_edge — Real-Time Stability Screening on the Edge", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex12.2_39_edge_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figure of section 6, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex12.2_39_edge_report.md into Ex12.2_39_edge_report.pdf, with any figure
# saved as Ex12.2_39_edge_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex12.2_39_edge_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex12.2_39_edge_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex12.2_39_edge_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex12.2_39_edge_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 10 · Mini project proposal

Two problems grown out of this set, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**, built by the course with a classical
solver and shown to have converged; the worked example below each project is
its first case. You do not solve the physics a second way: the month goes into
the deep learning - making a network predict what the ground truth says, as
accurately and as fast as the requirements ask.

### MP12.2A-39 · The real-time loop on the two boards

**The problem.** The set ran the loop on a virtual board. Here it runs on the hardware: the
Jetson AGX Orin plays the PMU stream, the Jetson AGX Thor screens it, and
the two talk over the lab network. The frame budget is real: 20 ms from the
moment a frame leaves the Orin to the moment its verdicts come back.

| | |
|---|---|
| recording | ten minutes of the grid at 50 frames a second (30 000 frames) |
| player | `board/pmu_player.py` on the Orin, unchanged |
| screener | `board/screener.py` on the Thor, yours to make fast |
| contingencies | the set's 35 |

**What you build.** The set's surrogate on the Thor, made fast enough: half precision, a compiled
graph (`torch.compile`, CUDA graphs or TensorRT), the fault phase moved onto
the surrogate if you can keep it accurate. You measure each step's share of
the time.

**The ground truth you get.** `MP12.2A-39_recording.npz` (what the Orin plays) and `MP12.2A-39_truth.npz`: the
reference verdicts of all 35 contingencies for one frame every second, by
RK4 on 10 ms, from `tools/miniprojects/ex122_39_edge_truth.py A`.

**Worked example.** Over the 600 checked frames the same two contingencies are insecure in every
frame: a fault at bus 10 cleared by tripping branch 10-11, and the same fault
cleared by tripping 10-13. In this recording the verdicts do not change with
the drifting load (they change with which units are converters), so the
project is a test of latency first. RK4 on 5 ms changes no verdict against
10 ms on 20 frames.

**Requirements.**

* Your verdicts agree with the truth on every checked frame, or each disagreement explained.
* Round trip under **20 ms** for 99 % of the frames, measured on the Orin's clock.
* No frame dropped by the screener over the ten minutes.

**What you hand in.** The latency of each part of the loop (network, fault phase, surrogate, the
answer), what you did to each, and the verdicts against the truth.

### MP12.2B-39 · More renewable power, and what converters do in a fault

**The problem.** The set kept converters below 45 % of the local power and let them behave like
small machines. At 50 to 80 % they set the grid's behaviour, and in a fault
they do what a machine cannot: they **limit their current**, delivering
little active power while the voltage is low, and ramp back afterwards. Their
inertia is small, so the grid's frequency moves faster.

| | |
|---|---|
| states | 20: five to eight of the nine local machines replaced (share 50 to 80 %), load 0.9 to 1.1 |
| converters | grid-following, inertia 0.5 s, $D/M$ = 1 1/s (assumed) |
| in a fault, *keep* | the converters keep delivering their power |
| in a fault, *limit* | every converter within two branches of the faulted bus delivers no active power, and ramps back over 0.5 s after clearing (assumed) |
| contingencies | the set's 35 |

**Why a plain PINN is not enough.** The converters' small inertia brings faster swings than a 50 ms step
follows, and the current limit is a switch the set's surrogate never saw.
Both must enter what you build.

**What you build.** One surrogate that screens these states for both behaviours, the behaviour an
input: retrained on trajectories you simulate, with a step and inputs you
choose, run on the Orin and Thor loop of MP12.2A.

**The ground truth you get.** `MP12.2B-39_truth.npz`, from `tools/miniprojects/ex122_39_edge_truth.py B`: every
contingency's largest separation for the 20 states and both behaviours, by
RK4 on 5 ms (checked against 2.5 ms on five of them).

**Worked example.** If the converters keep their power through the fault, 126 of the 700 cases are
insecure (18 %); if they limit it, none is. In this model the converters'
behaviour in a fault, not their share, decides the verdicts: a screen that
does not know what the converters do would be wrong on one case in six.
RK4 on 2.5 ms changes no verdict against 5 ms.

**Requirements.**

* Every insecure case caught in both behaviours; at most **5 %** false alarms in either.
* One state's screen within the **20 ms** frame on the Thor.
* Say why limiting the current makes the grid more secure here, and whether you expect that on a real grid.

**What you hand in.** Caught, missed and false alarms against the truth, the latency on the boards,
and what the current limit did to the screen.